# 제품별 검증과 검사 우선순위

제출 노트북에서 최종 검증 경로를 분리했습니다. 새 커널에서 위에서 아래로 실행합니다. 원천 데이터는 배포하지 않습니다. **이 공개용 구성은 실제 데이터로 재학습하지 않았으며, README 성능은 원본 저장 출력입니다.**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

# 저장소 루트 또는 notebooks/에서 실행
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
required = ["cn7_final.csv", "rg3_final.csv", "moldset_labeled.csv"]
missing = [name for name in required if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError("data/ 에 사용 권한이 있는 입력을 준비하세요: " + ", ".join(missing))
cn7 = pd.read_csv(DATA / "cn7_final.csv")
rg3 = pd.read_csv(DATA / "rg3_final.csv")
raw = pd.read_csv(DATA / "moldset_labeled.csv")
for df in (cn7, rg3):
    df["TimeStamp"] = pd.to_datetime(df["TimeStamp"], errors="raise")
    if not df["_id"].is_unique:
        raise ValueError("학습 데이터의 _id가 중복되었습니다.")
    if df["PassOrFail"].isna().any() or not set(df["PassOrFail"].unique()).issubset({0, 1}):
        raise ValueError("PassOrFail은 결측 없는 0/1이어야 합니다.")
rg3["Side_RH"] = (rg3["Side"] == "RH").astype(int)

process_features = ['Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']

if not raw["_id"].is_unique:
    raise ValueError("원천 데이터의 _id가 중복되었습니다.")
if not cn7["_id"].isin(raw["_id"]).all():
    raise ValueError("CN7과 원천 데이터의 ID 연결에 누락이 있습니다.")
cn7_raw = cn7[["_id"]].merge(raw[["_id"] + process_features], on="_id", how="left", validate="one_to_one")
# 제출 당시 학습용 시간 정의를 유지합니다. 추론과의 차이는 docs/limitations.md 참고.
cn7["Hour"] = cn7["TimeStamp"].dt.hour
cn7["Hour_sin"] = np.sin(2 * np.pi * cn7["Hour"] / 24)
cn7["Hour_cos"] = np.cos(2 * np.pi * cn7["Hour"] / 24)


## 데이터 구조와 공정 입력의 Label Conflict
공정값이 같은데 라벨이 다른 사례를 요약합니다. 이를 라벨 오류나 모든 입력에서의 예측 불가능성으로 단정하지 않습니다.

In [ ]:
rows = []
for name, df in [("CN7", cn7), ("RG3", rg3)]:
    grouped = df.groupby(process_features, dropna=False)["PassOrFail"].agg(["size", "sum", "nunique"])
    conflicts = grouped[grouped["nunique"] > 1]
    rows.append({"Product": name, "Rows": len(df), "Defects": int(df["PassOrFail"].sum()),
                 "Conflicting_process_groups": len(conflicts),
                 "Defects_in_conflicting_groups": int(conflicts["sum"].sum())})
display(pd.DataFrame(rows))
# 공정 24개가 같은 그룹 기준입니다. Side를 포함하면 구별 가능성이 달라집니다.


## CN7: 시간대와 직전 관측 Shot 변화량
원천 공정값을 ID로 연결합니다. 동일 TimeStamp를 한 Shot으로 간주합니다. 누락된 생산 사이클이 있다면 직전 관측 Shot이 실제 직전 사이클과 다를 수 있습니다.

In [ ]:
# ==========================================
# CN7 4차 모델링 ④
# 직전 Shot 대비 공정 변화량 생성
# ==========================================

change_cols = [
    "Max_Injection_Speed",
    "Max_Injection_Pressure",
    "Filling_Time",
    "Plasticizing_Time"
]

# cn7 행과 raw 값 결합
temp = cn7[
    ["_id", "TimeStamp"]
].copy()

for col in change_cols:
    temp[f"raw_{col}"] = cn7_raw[col].values

temp["TimeStamp"] = pd.to_datetime(temp["TimeStamp"])


# ------------------------------------------
# 동일 TimeStamp = 동일 shot
# 대표값 하나만 남김
# ------------------------------------------

raw_change_cols = [
    f"raw_{col}" for col in change_cols
]

shot_df = (
    temp
    .sort_values("TimeStamp")
    .groupby("TimeStamp", as_index=False)[raw_change_cols]
    .first()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# ------------------------------------------
# 직전 shot 대비 변화량
# ------------------------------------------

for col in change_cols:
    shot_df[f"Delta_{col}"] = (
        shot_df[f"raw_{col}"].diff()
    )


delta_cols = [
    f"Delta_{col}" for col in change_cols
]


# 첫 shot은 이전 shot이 없으므로 0
shot_df[delta_cols] = shot_df[delta_cols].fillna(0)


# ------------------------------------------
# 원래 CN7 행에 다시 붙이기
# ------------------------------------------

cn7 = cn7.merge(
    shot_df[["TimeStamp"] + delta_cols],
    on="TimeStamp",
    how="left",
    validate="many_to_one"
)

print(cn7[delta_cols].describe().round(3))
print()
print("Missing:")
print(cn7[delta_cols].isna().sum())

## CN7: TimeStamp 그룹 4-fold OOF 평가
같은 TimeStamp의 행은 같은 fold에 배정합니다. 시간 순서 분할이 아니며, 같은 개발 검증에서 모델·특징을 비교한 선택 편향이 남습니다. Delta는 라벨을 사용하지 않지만 분할 전 전체 관측 이력으로 계산합니다. 이 평가가 미래 기간 일반화나 모든 종류의 누수 방지를 입증하지는 않습니다.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score


# ------------------------------------------
# 최종 Feature
# ------------------------------------------

cyclic_features = [
    "Hour_sin",
    "Hour_cos"
]

delta_features = [
    "Delta_Max_Injection_Speed",
    "Delta_Max_Injection_Pressure",
    "Delta_Filling_Time",
    "Delta_Plasticizing_Time"
]

final_features = (
    process_features
    + cyclic_features
    + delta_features
)

X = cn7[final_features]
y = cn7["PassOrFail"].astype(int)

groups = cn7["TimeStamp"].astype(str)


# ------------------------------------------
# 기존과 동일한 Grouped CV
# ------------------------------------------

cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)


# ------------------------------------------
# 최종 ExtraTrees
# ------------------------------------------

final_model = ExtraTreesClassifier(
    n_estimators=700,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------
# OOF Prediction — 개발 검증, 미래 기간 테스트가 아님
# ------------------------------------------

for train_idx, valid_idx in cv.split(X, y, groups):
    if set(groups.iloc[train_idx]) & set(groups.iloc[valid_idx]):
        raise ValueError("학습·검증 Shot 그룹이 겹칩니다.")

cn7_final_prob = cross_val_predict(
    final_model,
    X,
    y,
    cv=cv,
    groups=groups,
    method="predict_proba",
    n_jobs=-1
)[:, 1]


print("ROC-AUC :", round(roc_auc_score(y, cn7_final_prob), 3))
print("AP      :", round(average_precision_score(y, cn7_final_prob), 3))

In [ ]:
inspection_rates = np.arange(0.01, 0.31, 0.01)

total_defects = y.sum()
n = len(y)

budget_results = []

for rate in inspection_rates:

    k = int(np.ceil(n * rate))

    # 위험도가 높은 순서대로 검사
    top_idx = np.argsort(cn7_final_prob)[::-1][:k]

    caught_defects = y.iloc[top_idx].sum()

    recall = caught_defects / total_defects

    budget_results.append({
        "Inspection_Rate": rate,
        "Inspection_Count": k,
        "Caught_Defects": int(caught_defects),
        "Total_Defects": int(total_defects),
        "Recall": recall
    })


budget_df = pd.DataFrame(budget_results)

display(budget_df)

## RG3: 이후 날짜 평가
2020-10-23 이전 학습, 그날 이후 평가입니다. 이 구간도 여러 실험에 사용되어 완전히 독립적인 최종 시험으로 해석하지 않습니다.

In [ ]:
rg3["TimeStamp"] = pd.to_datetime(rg3["TimeStamp"])

rg3_train = rg3[
    rg3["TimeStamp"] < "2020-10-23"
].copy()

rg3_test = rg3[
    rg3["TimeStamp"] >= "2020-10-23"
].copy()

print("===== RG3 =====")
print(
    "Train:",
    len(rg3_train),
    "불량:",
    rg3_train["PassOrFail"].sum()
)

print(
    "Test:",
    len(rg3_test),
    "불량:",
    rg3_test["PassOrFail"].sum()
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 최종 RG3 feature
# 24개 공정변수 + Side_RH
rg3_final_features = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
    "Side_RH"
]

X_train = rg3_train[rg3_final_features]
y_train = rg3_train["PassOrFail"]

X_test = rg3_test[rg3_final_features]
y_test = rg3_test["PassOrFail"]

# RG3 최종 RandomForest
rf_rg3_final = RandomForestClassifier(
    n_estimators=700,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_rg3_final.fit(X_train, y_train)

# chronological test 500건 예측확률
rg3_test_prob = rf_rg3_final.predict_proba(X_test)[:, 1]

print("Train:", X_train.shape, "Defects:", y_train.sum())
print("Test :", X_test.shape, "Defects:", y_test.sum())

In [ ]:
# 위험도 높은 순으로 정렬
order = np.argsort(-rg3_test_prob)

y_true = y_test.to_numpy()
y_sorted = y_true[order]

total_defects = y_true.sum()
n = len(y_true)

budget_results = []

# 검사 비율 1% ~ 30%
for rate in np.arange(0.01, 0.31, 0.01):
    inspected = int(np.ceil(n * rate))
    detected = int(y_sorted[:inspected].sum())

    budget_results.append({
        "Inspection_Rate": round(rate, 2),
        "Inspected": inspected,
        "Detected_Defects": detected,
        "Recall": detected / total_defects,
        "Precision": detected / inspected
    })

rg3_budget = pd.DataFrame(budget_results)

rg3_budget[
    rg3_budget["Inspection_Rate"].isin([0.05, 0.10, 0.20])
]

## 해석
CN7 OOF 결과와 RG3 시간 분할 결과는 검증 방식이 다릅니다. 모델 점수는 제품 내 상대적 순위이며, 보정된 불량 확률이 아닙니다. 노트북을 다시 저장할 때 행 단위 출력과 데이터가 포함되지 않았는지 확인하세요.